# 10.1 上下文扩展 (Context Extension)

大模型默认上下文窗口有限（2K-8K），上下文扩展技术使模型能处理更长的输入序列，是长上下文处理的基础。

## 上下文扩展方法总览

| 方法 | 原理 | 最大长度 | 是否需要微调 | 代表工作 |
|------|------|----------|-------------|----------|
| 位置插值(PI) | 缩放位置编码 | 16K | 是 | Meta |
| NTK-Aware | 调整RoPE基频 | 32K | 否/是 | Reddit |
| YaRN | 动态缩放+注意力调整 | 128K | 是 | YaRN |
| ALiBi | 线性偏置注意力 | 无限 | 否 | Press et al. |
| LongRoPE | 非均匀位置插值 | 128K | 是 | LongRoPE |

## 1. RoPE位置编码回顾

RoPE（Rotary Position Embedding）是当前LLM最常用的位置编码方式，理解RoPE是理解上下文扩展的基础。

### RoPE核心公式
$$q_m = q \odot [\cos(m\theta_0), \sin(m\theta_0), \cos(m\theta_1), \sin(m\theta_1), ...]$$
$$k_n = k \odot [\cos(n\theta_0), \sin(n\theta_0), \cos(n\theta_1), \sin(n\theta_1), ...]$$

其中 $\theta_i = 10000^{-2i/d}$，$m$和$n$是位置索引。

### RoPE的关键性质
- 注意力分数只依赖相对位置$m-n$
- 远距离token的注意力自然衰减
- 超出训练长度时，模型遇到未见过的位置，性能急剧下降

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class RoPEEncoding(nn.Module):
    def __init__(self, d_model=64, max_len=2048, base=10000):
        super().__init__()
        self.d_model = d_model
        self.base = base
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, x, seq_len=None):
        if seq_len is None:
            seq_len = x.shape[1]
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos().unsqueeze(0)
        sin = emb.sin().unsqueeze(0)
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        rotated = torch.cat([x1 * cos[..., :x1.shape[-1]] - x2 * sin[..., :x2.shape[-1]],
                             x1 * sin[..., :x1.shape[-1]] + x2 * cos[..., :x2.shape[-1]]], dim=-1)
        return rotated

rope = RoPEEncoding(d_model=64, base=10000)
x = torch.randn(1, 128, 64)
x_rotated = rope(x, seq_len=128)

print('=== RoPE Position Encoding ===')
print(f'Input: {x.shape}')
print(f'Rotated: {x_rotated.shape}')
print(f'Base: {rope.base}')
print(f'Inv freq shape: {rope.inv_freq.shape}')

with torch.no_grad():
    q = x_rotated[0, :1, :]
    for dist in [1, 10, 50, 100]:
        k = x_rotated[0, dist:dist+1, :]
        attn = (q @ k.T).item() / math.sqrt(64)
        print(f'  Distance {dist:3d}: attention score = {attn:.4f}')

print(f'\nKey: RoPE encodes relative position via rotation.')
print(f'Attention naturally decays with distance.')

## 2. 位置插值（Position Interpolation, PI）

位置插值是最直接的上下文扩展方法：将长序列的位置索引线性映射到训练时的位置范围内。

### 核心思想
- 训练时模型见过位置0~L-1
- 推理时序列长度L' > L
- 将位置索引从[0, L']映射到[0, L]，即$m' = m \cdot L / L'$
- 模型看到的位置始终在训练范围内

### 优缺点
- **优点**：简单、有效、只需少量微调
- **缺点**：线性缩放导致近距离分辨率降低，远距离信息可能模糊

In [ ]:
class PositionInterpolationRoPE(nn.Module):
    def __init__(self, d_model=64, base=10000, original_max_len=2048, target_max_len=8192):
        super().__init__()
        self.d_model = d_model
        self.base = base
        self.original_max_len = original_max_len
        self.target_max_len = target_max_len
        self.scale = original_max_len / target_max_len
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, x, seq_len=None):
        if seq_len is None:
            seq_len = x.shape[1]
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        t_scaled = t * self.scale
        freqs = torch.outer(t_scaled, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos().unsqueeze(0)
        sin = emb.sin().unsqueeze(0)
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        rotated = torch.cat([x1 * cos[..., :x1.shape[-1]] - x2 * sin[..., :x2.shape[-1]],
                             x1 * sin[..., :x1.shape[-1]] + x2 * cos[..., :x2.shape[-1]]], dim=-1)
        return rotated

class NTKAwareRoPE(nn.Module):
    def __init__(self, d_model=64, base=10000, original_max_len=2048, target_max_len=8192):
        super().__init__()
        self.d_model = d_model
        scale = target_max_len / original_max_len
        new_base = base * (scale ** (d_model / (d_model - 2)))
        self.base = new_base
        inv_freq = 1.0 / (new_base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, x, seq_len=None):
        if seq_len is None:
            seq_len = x.shape[1]
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos().unsqueeze(0)
        sin = emb.sin().unsqueeze(0)
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        rotated = torch.cat([x1 * cos[..., :x1.shape[-1]] - x2 * sin[..., :x2.shape[-1]],
                             x1 * sin[..., :x1.shape[-1]] + x2 * cos[..., :x2.shape[-1]]], dim=-1)
        return rotated

x = torch.randn(1, 8192, 64)

rope_orig = RoPEEncoding(d_model=64, base=10000)
rope_pi = PositionInterpolationRoPE(d_model=64, base=10000, original_max_len=2048, target_max_len=8192)
rope_ntk = NTKAwareRoPE(d_model=64, base=10000, original_max_len=2048, target_max_len=8192)

x_orig = rope_orig(x, seq_len=8192)
x_pi = rope_pi(x, seq_len=8192)
x_ntk = rope_ntk(x, seq_len=8192)

print('=== Context Extension Methods ===')
print(f'Original RoPE base: 10000')
print(f'PI scale factor: {rope_pi.scale:.4f} (positions compressed)')
print(f'NTK new base: {rope_ntk.base:.0f} (base frequency adjusted)')

with torch.no_grad():
    q_pos = 0
    print(f'\nAttention scores from position {q_pos}:')
    print(f'{"Distance":>10} {"Original":>10} {"PI":>10} {"NTK-Aware":>10}')
    for dist in [1, 100, 1000, 4000, 8000]:
        if dist < 8192:
            score_orig = (x_orig[0, q_pos] @ x_orig[0, dist]).item() / math.sqrt(64)
            score_pi = (x_pi[0, q_pos] @ x_pi[0, dist]).item() / math.sqrt(64)
            score_ntk = (x_ntk[0, q_pos] @ x_ntk[0, dist]).item() / math.sqrt(64)
            print(f'{dist:>10} {score_orig:>10.4f} {score_pi:>10.4f} {score_ntk:>10.4f}')

print(f'\nKey: PI compresses positions, NTK adjusts base frequency.')
print(f'NTK preserves local resolution better than PI.')
print(f'Both need fine-tuning for best results.')

## 3. ALiBi（Attention with Linear Biases）

ALiBi不使用位置编码，而是在注意力分数上添加与距离成比例的线性偏置，天然支持长度外推。

### 核心公式
$$\text{attention}(q_i, k_j) = q_i^T k_j - m \cdot |i - j|$$

其中$m$是每个注意力头的斜率，与头数相关。

### ALiBi优势
- **无需位置编码**：简化模型架构
- **天然外推**：训练2K，推理可以到32K+
- **线性衰减**：远距离token注意力自然降低

### ALiBi局限
- 性能略低于RoPE（在训练长度内）
- 线性衰减可能过于激进，丢失远距离信息
- 当前主流LLM（LLaMA、GPT等）仍使用RoPE

In [ ]:
class ALiBiAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4, max_len=8192):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        slopes = torch.tensor([2 ** (-8 * i / n_heads) for i in range(n_heads)])
        self.register_buffer('slopes', slopes)

        positions = torch.arange(max_len).unsqueeze(0) - torch.arange(max_len).unsqueeze(1)
        bias = positions.abs().unsqueeze(0) * slopes.unsqueeze(1).unsqueeze(2)
        self.register_buffer('bias', -bias)

    def forward(self, x):
        B, T, D = x.shape
        q = self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = attn + self.bias[:, :T, :T]
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, D)
        return self.out_proj(out)

alibi = ALiBiAttention(d_model=64, n_heads=4, max_len=8192)
x = torch.randn(2, 128, 64)
out = alibi(x)

print('=== ALiBi Attention ===')
print(f'Input: {x.shape}')
print(f'Output: {out.shape}')
print(f'\nSlopes per head: {[f"{s:.6f}" for s in alibi.slopes.tolist()]}')

print(f'\nAttention bias at distance d (head 0, slope={alibi.slopes[0]:.6f}):')
for d in [1, 10, 100, 1000, 8000]:
    bias = -alibi.slopes[0].item() * d
    print(f'  Distance {d:5d}: bias = {bias:.4f}')

print(f'\nKey: ALiBi adds linear distance penalty to attention.')
print(f'No position embedding needed, naturally extrapolates to longer sequences.')
print(f'Slopes decrease geometrically across heads for multi-scale distance sensing.')

## 4. 上下文扩展工业实践

### 方法选择指南
| 场景 | 推荐方法 | 原因 |
|------|----------|------|
| 快速扩展2x-4x | NTK-Aware (无微调) | 零样本扩展，最简单 |
| 稳定扩展4x-8x | PI + 微调 | 可靠，需要少量长文本数据 |
| 大幅扩展8x+ | YaRN + 微调 | 最先进，需要更多微调数据 |
| 新模型设计 | ALiBi | 天然外推，但RoPE更主流 |

### 微调数据需求
- **PI微调**：~1000条长文本，1000步即可
- **YaRN微调**：~5000条长文本，2000步
- **关键**：微调数据长度应覆盖目标长度的80%+

### 最佳实践
1. **渐进式扩展**：4K→8K→16K→32K，而不是直接跳到目标长度
2. **混合长度训练**：同时使用短文本和长文本，防止短文本能力退化
3. **评估验证**：用Needle-in-a-Haystack测试验证长上下文能力
4. **注意力温度**：YaRN中调整注意力温度补偿远距离注意力衰减
5. **工程优化**：长上下文训练需要FlashAttention和梯度检查点